# CAPE-TTA V4 — 候选多样性诊断

**操作：代码执行程序 → 更改运行时类型 → T4 GPU → 保存，然后点击“全部运行”。**

本次查明 V3 中 4 个候选完全相同的原因。固定同一初始观测，比较原采样设置
与关闭 top-p/top-k 截断、不同温度的预设开发设置，记录概率支持、熵和动作重复率。
默认只运行 Spatial/task 0/init 0，4 组设置各 4 个 seed，并检查一次同 seed 重复。
不强制产生独立候选，不训练、不执行任务 rollout，也不据此声称论文方法有效。

- 自动检测可复用环境；新运行时会先安装依赖。旧运行时的缓存不一定能跨 notebook 复用。
- 使用固定源代码 7715fa660da41bdb60a07f18132c74bc2ac1ab9a，模型 checkpoint 固定为 V3 版本。
- 只加载一次模型，页面输出日志时不要重复点击运行。
- 结束后自动下载 ZIP；成功或失败都把 ZIP 发回本对话。
- 若下载被浏览器阻止，允许本页下载后，在左侧文件中找到 /content/cape-diversity 下的 ZIP。
- 本 notebook 已做语法与模拟故障测试；本次 GPU 诊断尚未执行。


In [ ]:
# V4 sampling diagnostic: one cell, automatic environment setup and ZIP download.
import datetime as dt
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import traceback
import uuid

SOURCE_SHA = "7715fa660da41bdb60a07f18132c74bc2ac1ab9a"
SOURCE = Path("/content/cape-source") / SOURCE_SHA
ROOT = Path("/content/cape-diversity") / ("diagnostic-" + dt.datetime.now().strftime("%Y%m%d-%H%M%S") + "-" + uuid.uuid4().hex[:8])
ROOT.mkdir(parents=True, exist_ok=False)
PY = Path("/content/cape-env-v3/bin/python")
LIBERO = Path("/content/LIBERO-8f1084e3132a")
state = {"status": "running", "scientific_result": False, "source_sha": SOURCE_SHA}
log = (ROOT / "launcher.log").open("w", encoding="utf-8")

def run(command, env=None):
    line = "$ " + " ".join(map(str, command))
    print(line, flush=True)
    log.write(line + "\n"); log.flush()
    with subprocess.Popen(list(map(str, command)), stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1, env=env) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
            log.write(line); log.flush()
        status = proc.wait()
    if status:
        raise subprocess.CalledProcessError(status, command)

try:
    smi = shutil.which("nvidia-smi")
    if smi is None:
        raise RuntimeError("请选择：代码执行程序 → 更改运行时类型 → T4 GPU → 保存，再运行。")
    run([smi, "--query-gpu=name,memory.total", "--format=csv,noheader"])
    if not SOURCE.exists():
        SOURCE.mkdir(parents=True)
        run(["git", "-C", SOURCE, "init"])
        run(["git", "-C", SOURCE, "remote", "add", "origin", "https://github.com/weichezhang/CAPE-TTA.git"])
        run(["git", "-C", SOURCE, "fetch", "--depth", "1", "origin", SOURCE_SHA])
        run(["git", "-C", SOURCE, "checkout", "--detach", SOURCE_SHA])
    actual = subprocess.check_output(["git", "-C", str(SOURCE), "rev-parse", "HEAD"], text=True).strip()
    if actual != SOURCE_SHA:
        raise RuntimeError("Unexpected source revision: " + actual)
    subprocess.run(["git", "-C", str(SOURCE), "diff", "--quiet", "HEAD"], check=True)
    warm = False
    if PY.is_file() and LIBERO.is_dir():
        check = subprocess.run([str(PY), "-c", "import torch, transformers, bitsandbytes, mujoco; assert torch.cuda.is_available(); assert mujoco.__version__ == '2.3.7'; assert torch.__version__.startswith('2.2.0')"],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        warm = check.returncode == 0
        log.write(check.stdout); log.flush()
    if not warm:
        print("准备运行环境；本步骤不会重复 V3 分支实验。", flush=True)
        run([sys.executable, "-u", SOURCE / "scripts/colab_bootstrap.py", "--prepare-only", "--run-dir", ROOT / "setup"])
    else:
        print("检测到可复用的 V3 环境。", flush=True)
    # Install the pinned diagnostic code instead of reusing the old editable package.
    run([PY, "-m", "pip", "install", "--no-build-isolation", "--no-deps", "-e", SOURCE])
    env = dict(os.environ, HF_HOME="/content/cape-hf-cache", PYTHONUNBUFFERED="1",
               MUJOCO_GL="osmesa", PYOPENGL_PLATFORM="osmesa", TF_CPP_MIN_LOG_LEVEL="2")
    run([PY, "-u", SOURCE / "scripts/probe_openvla_diversity.py",
         "--libero-root", LIBERO, "--output-root", ROOT / "probes"], env=env)
    state["status"] = "diagnostic_complete"
    print("诊断完成；它只检查候选分布，不代表任务成功率提升。", flush=True)
except BaseException as exc:
    state["status"] = "failed"
    state["error"] = {"type": type(exc).__name__, "message": str(exc), "traceback": traceback.format_exc()}
    print(traceback.format_exc(), flush=True)
    raise
finally:
    (ROOT / "launcher_status.json").write_text(json.dumps(state, indent=2), encoding="utf-8")
    log.close()
    archive = shutil.make_archive(str(ROOT), "zip", ROOT)
    print("\n请把这个 ZIP 上传回对话：", archive, flush=True)
    from google.colab import files
    files.download(archive)
